# SARM Reward Model Training — Lite

精簡版訓練 notebook：只保留訓練必要的 cell（GPU 檢查 → 設定 → 安裝 → patch → 登入 → 訓練）。

完整版（含資料集預覽、視覺化、一次性修補）請看 `SARM_Training_Colab.ipynb`。
訓練後想跑預測視覺化請看 `SARM_Predict_Visualization.ipynb`。

In [ ]:
# Cell 1: GPU 確認
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 70:
    print('WARNING: VRAM < 70GB，建議使用 A100 80GB')
else:
    print('✓ GPU 規格符合需求')

In [ ]:
# Cell 2: 使用者設定（必填）
# ============================================================
HF_USERNAME = 'Lebruhbruh'
HF_TOKEN    = ''   # <-- 貼上你的 Write Token

# 已完成標注+修正的資料集（直接用於訓練，不需要再 merge / 標注）
#   - 273 episodes / 585,425 frames
#   - 來源：AllenAI charging-01~06（共 284 eps，排除 11 個問題 eps）
#   - 標注：Pick up phone → Flip sideways → Plug cable → Turn on power
#   - 相機：top / left / right 三個視角
#   - subtask_index 0→1→2→3 在每個 episode 內單調遞增
#   - v3.0 git tag 已建立，stats.json 已重算
ANNOTATED_DATASET = 'Lebruhbruh/SARM-opendata-annotated-fixed'

# 訓練好的 SARM 模型 repo
MODEL_REPO_ID = f'{HF_USERNAME}/sarm-charging-bimanual'

# 主要訓練相機
VIDEO_KEY = 'observation.images.top'

# 四個子任務（已標注在 ANNOTATED_DATASET 中，subtask_index 0–3）
SUBTASK_NAMES = [
    'Pick up the phone',
    'Flip the phone sideways',
    'Pick up the charging cable and plug it into the phone',
    'Turn on the power of the extension cord',
]

print(f'標注資料集: {ANNOTATED_DATASET}')
print(f'SARM 模型:  {MODEL_REPO_ID}')
print(f'相機:       {VIDEO_KEY}')
print('子任務:')
for i, t in enumerate(SUBTASK_NAMES):
    print(f'  [{i}] {t}')

In [ ]:
# Cell 3: 進度回饋工具（後續長時 cell 共用）
# ============================================================
# progress_stage(title, steps, heartbeat_seconds): context manager
#   - 開頭印 ━━━ 標題 ━━━ 與步驟清單
#   - 背景 thread 每 N 秒印一次 ⏱ 心跳（含已經過時間）
#   - 結尾印 ✓ 與總耗時
# 預期被 Cell 2 / 4b / 9a / 10 / 12 / 13 / 14 等長時 cell 使用。
# ============================================================
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()

    stop_evt = threading.Event()
    start = time.monotonic()

    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)

    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')


In [ ]:
# Cell 4: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認 lerobot / transformers 版本'],
    heartbeat_seconds=30,
):
    print('1/2 安裝 LeRobot + SARM...')
    # CLAUDE.md 規則 4：>30 秒指令不可 capture_output；讓 pip 即時輸出
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗，返回碼 {r.returncode}')
    print('  ✓ LeRobot 安裝完成')

    print('2/2 確認安裝...')
    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    if r.returncode == 0:
        versions = r.stdout.strip().split()
        print(f'  LeRobot: {versions[0]}')
        print(f'  Transformers: {versions[1] if len(versions)>1 else "?"}')
    else:
        raise RuntimeError(f'版本確認失敗: {r.stderr}')


In [ ]:
# Cell 5: 修補 Transformers 5.x Bug（CRITICAL）
import os, glob, torch, subprocess
import lerobot

lerobot_root = os.path.dirname(lerobot.__file__)
print(f"LeRobot 安裝路徑: {lerobot_root}")

hits = glob.glob(os.path.join(lerobot_root, "**", "processor_sarm.py"), recursive=True)
if not hits:
    print("lerobot_root 下找不到，往上一層搜尋...")
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), "**", "processor_sarm.py"), recursive=True)

if not hits:
    raise FileNotFoundError(
        "找不到 processor_sarm.py，請確認 lerobot[sarm] 安裝成功。"
        "可在新 cell 執行: !find /usr/local/lib -name processor_sarm.py"
    )

processor_path = hits[0]
print(f"修補目標: {processor_path}")

with open(processor_path) as f:
    content = f.read()

patched = False

indent = "\n        "
old_img = "embeddings = self.clip_model.get_image_features(**inputs).detach().cpu()"
new_img = indent.join([
    "output = self.clip_model.get_image_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_img in content:
    content = content.replace(old_img, new_img)
    print("✓ 已修補 image features")
    patched = True
else:
    print("ℹ image features 不需修補")

old_txt = "embeddings = self.clip_model.get_text_features(**inputs).detach().cpu()"
new_txt = indent.join([
    "output = self.clip_model.get_text_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_txt in content:
    content = content.replace(old_txt, new_txt)
    print("✓ 已修補 text features")
    patched = True
else:
    print("ℹ text features 不需修補")

with open(processor_path, "w") as f:
    f.write(content)

if patched:
    r = subprocess.run(["grep", "-n", "pooler_output", processor_path],
                       capture_output=True, text=True)
    print("\n驗證修補（應看到 pooler_output）:")
    print(r.stdout)
else:
    print("\n✓ 程式碼已是最新版，無需修補")


In [ ]:
# Cell 6: HuggingFace 登入
from huggingface_hub import login
login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')

In [ ]:
# Cell 7: 訓練 SARM Reward Model（預計 45-90 分鐘）
# ============================================================
# SARM 是 RewardModelConfig（不是 Policy），所以 lerobot-train 用
#   --reward_model.type=sarm
# 而不是 --policy.type=sarm。
# CLI 格式：draccus（帶 `--`，巢狀用 `.` 分隔）
# ============================================================
import subprocess, os, sys, shutil

env = os.environ.copy()
env.update({
    'HF_TOKEN': HF_TOKEN,
    'HUGGING_FACE_HUB_TOKEN': HF_TOKEN,
    'PYTHONUNBUFFERED': '1',   # 子程序 stdout/stderr 不要 buffer，立刻吐出來
})

lerobot_bin = shutil.which('lerobot-train')
lerobot_train = [lerobot_bin] if lerobot_bin else [sys.executable, '-m', 'lerobot.scripts.lerobot_train']
print(f'binary: {lerobot_train}')

# Fix #2: 避免 FileExistsError — lerobot 在 output_dir 已存在且 resume=False 時會拋錯
out_dir = 'outputs/train/sarm_charging_bimanual'
if os.path.isdir(out_dir):
    print(f'⚠ 清掉舊的 {out_dir}（避免 FileExistsError）')
    shutil.rmtree(out_dir, ignore_errors=True)

cmd = [
    *lerobot_train,
    # ── dataset ─────────────────────────────────────────────
    f'--dataset.repo_id={ANNOTATED_DATASET}',
    # ── reward model (SARM) ─────────────────────────────────
    '--reward_model.type=sarm',
    '--reward_model.annotation_mode=dense_only',
    f'--reward_model.image_key={VIDEO_KEY}',
    '--reward_model.state_key=observation.state',
    '--reward_model.n_obs_steps=8',
    '--reward_model.frame_gap=30',   # 30fps，gap=30 → 1 second context
    f'--reward_model.repo_id={MODEL_REPO_ID}',
    '--reward_model.push_to_hub=true',
    # ── top-level training ───────────────────────────────────
    f'--output_dir={out_dir}',
    '--batch_size=64',
    '--steps=5000',
    '--save_freq=2500',
    '--tolerance_s=0.001',           # 放寬 video timestamp 容差到 1ms（預設 1e-4 卡浮點漂移）
    '--wandb.enable=false',
]

print('命令:', ' '.join(cmd))

# 把所有輸出同時印到畫面跟 log 檔（Colab 重連也能事後看）
log_path = '/content/sarm_train.log'
print(f'log: {log_path}\n')

with progress_stage(
    '訓練 SARM Reward Model（5000 steps，預計 45-90 分鐘）',
    steps=[
        f'載入 {ANNOTATED_DATASET}（273 eps / 585,425 frames）',
        '初始化 SARMConfig（CLIP backbone + dense head，annotation_mode=dense_only）',
        '訓練 5000 steps（lerobot-train 即時印 step 進度）',
        f'推送模型到 HF Hub: {MODEL_REPO_ID}',
    ],
    heartbeat_seconds=30,
):
    # Fix #3 (v2): Popen + 逐行 read + Python print
    # 在 Colab/Jupyter 裡 subprocess 直接寫 FD 常被吞掉，
    # 改成 stdout=PIPE+stderr=STDOUT 一起接，再用 print() 走 Jupyter 通道，
    # 保證每一行（包含 traceback、tqdm 進度）都看得到。
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    with open(log_path, 'w') as logf:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
    returncode = proc.wait()

if returncode == 0:
    print(f'\n✓ 訓練完成！模型已上傳至: https://huggingface.co/{MODEL_REPO_ID}')
else:
    print(f'\n✗ 訓練失敗（返回碼: {returncode}）')
    print(f'   完整輸出已存到 {log_path}（可用 !tail -200 {log_path} 再看）')
